In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy joblib

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
data_path = "/kaggle/input/datasets/overgame1234/multilabel-tags-merged/multilabel_tags_merged.csv"


df = pd.read_csv(data_path)



In [3]:
df = df[["Title", "Tags"]].copy()
df = df.dropna(subset=["Title", "Tags"])

df["Title"] = df["Title"].astype(str).str.strip()
df["Tags"] = df["Tags"].astype(str).str.strip()

df = df[(df["Title"] != "") & (df["Tags"] != "")]



In [ ]:
df["label_list"] = df["Tags"].apply(lambda x: x.split())
df[["Title", "Tags", "label_list"]].head()  

,Title,Tags,label_list
0,# + items .append is not a function,javascript,[javascript]
1,# - how to parallel code that lock several obj...,c#,[c#]
2,# . what do and # do in this code,javascript,[javascript]
3,# .dialog is not a function error,javascript,[javascript]
4,# .dialog is not a function error after using ...,javascript,[javascript]


In [5]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 50
y shape: (1866458, 50)
First labels: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing']


In [6]:
X = df["Title"].tolist()

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", len(X_train), y_train.shape)
print("Val  :", len(X_val), y_val.shape)
print("Test :", len(X_test), y_test.shape)

Train: 1493166 (1493166, 50)
Val  : 186646 (186646, 50)
Test : 186646 (186646, 50)


In [8]:
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_size = min(650000, len(X_train))
val_size = min(50000, len(X_val))
test_size = min(50000, len(X_test))

train_idx = np.random.RandomState(42).choice(len(X_train), train_size, replace=False)
val_idx = np.random.RandomState(42).choice(len(X_val), val_size, replace=False)
test_idx = np.random.RandomState(42).choice(len(X_test), test_size, replace=False)

train_df = pd.DataFrame({
    "title": X_train.iloc[train_idx].values if hasattr(X_train, "iloc") else np.array(X_train)[train_idx],
    "labels": list(y_train[train_idx])
})

val_df = pd.DataFrame({
    "title": X_val.iloc[val_idx].values if hasattr(X_val, "iloc") else np.array(X_val)[val_idx],
    "labels": list(y_val[val_idx])
})

test_df = pd.DataFrame({
    "title": X_test.iloc[test_idx].values if hasattr(X_test, "iloc") else np.array(X_test)[test_idx],
    "labels": list(y_test[test_idx])
})
print(y_train.dtype, y_val.dtype, y_test.dtype)

float32 float32 float32


In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['title', 'labels'],
    num_rows: 650000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})


In [10]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["title"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 50


In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    jaccard = jaccard_score(labels, preds, average="samples", zero_division=0)

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "jaccard": jaccard
    }

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [16]:
trainer.train(resume_from_checkpoint="/kaggle/working/distilbert_results/checkpoint-20313")


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Jaccard
2,0.057973,0.059036,0.648813,0.767544,0.708421
3,0.053534,0.058456,0.652954,0.770366,0.713608


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=60939, training_loss=0.03758918628120789, metrics={'train_runtime': 6000.8643, 'train_samples_per_second': 324.953, 'train_steps_per_second': 10.155, 'total_flos': 3.2316568128e+16, 'train_loss': 0.03758918628120789, 'epoch': 3.0})

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
import torch

optimizer_state = torch.load("/kaggle/working/distilbert_results/checkpoint-18750/optimizer.pt")
print(type(optimizer_state))
print(optimizer_state.keys())

In [17]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.05845590680837631, 'eval_f1_macro': 0.6529541829134782, 'eval_f1_micro': 0.7703656127443825, 'eval_jaccard': 0.713608, 'eval_runtime': 79.7274, 'eval_samples_per_second': 627.137, 'eval_steps_per_second': 19.604, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.05895102396607399, 'eval_f1_macro': 0.6467297936346413, 'eval_f1_micro': 0.7674839462930532, 'eval_jaccard': 0.7097986666666666, 'eval_runtime': 80.403, 'eval_samples_per_second': 621.867, 'eval_steps_per_second': 19.44, 'epoch': 3.0}
